In [1]:
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

SEED = 42

In [2]:
df = pd.read_csv("olist_delivery_model_dataset.csv")

cat_cols = ["customer_state", "seller_state", "main_product_category"]
num_cols = ['customer_zip_code_prefix', 'n_items', 'n_unique_products', 'n_sellers',
            'total_price', 'total_freight', 'total_weight_g', 'max_item_weight_g',
            'total_volume_cm3', 'avg_photos_qty', 'avg_description_length',
            'freight_ratio', 'seller_zip_code_prefix', 'purchase_month',
            'purchase_dayofweek', 'purchase_hour', 'is_weekend',
            'same_state', 'same_city']
feature_cols = num_cols + cat_cols

X, y = df[feature_cols], df["delivery_days"]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [3]:
def make_model():
    prep = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)])
    return Pipeline([("prep", prep),
                     ("model", HistGradientBoostingRegressor(learning_rate=0.1, max_leaf_nodes=63,
                                                             max_iter=200, random_state=SEED))])

model = make_model().fit(x_train, y_train)
pred = model.predict(x_test)
print(f"Test set, with the real order details: MAE {mean_absolute_error(y_test, pred):.2f} days | "
      f"RMSE {root_mean_squared_error(y_test, pred):.2f} | R2 {r2_score(y_test, pred):.2f}")

Test set, with the real order details: MAE 4.67 days | RMSE 7.48 | R2 0.36


In [4]:
def typical_values(rows):
    by_category = rows.groupby("main_product_category")
    return {
        "product": by_category[["total_price", "total_weight_g", "max_item_weight_g", "total_volume_cm3",
                                "avg_photos_qty", "avg_description_length"]].median(),
        "seller": by_category[["seller_state", "seller_zip_code_prefix"]].agg(lambda s: s.mode()[0]),
        "state": rows.groupby("customer_state")[["customer_zip_code_prefix", "total_freight"]].median(),
    }

def make_row(t, category, state, month, dayofweek, hour):
    # the 22 model features for one (category, state, time of click)
    product = t["product"].loc[category].to_dict()
    seller = t["seller"].loc[category].to_dict()
    place = t["state"].loc[state].to_dict()
    return {**product, **seller, **place,
            "n_items": 1, "n_unique_products": 1, "n_sellers": 1,
            "freight_ratio": place["total_freight"] / product["total_price"],
            "purchase_month": month, "purchase_dayofweek": dayofweek, "purchase_hour": hour,
            "is_weekend": int(dayofweek >= 5), "same_state": int(state == seller["seller_state"]), "same_city": 0,
            "customer_state": state, "main_product_category": category}

In [6]:
mae_simple = mean_absolute_error(y_test, pred_simple)


NameError: name 'pred_simple' is not defined

In [8]:
final_model = make_model().fit(X, y)
bundle = dict(model=final_model, typical=typical_values(df), typical_error_days=5.3)
joblib.dump(bundle, "delivery_model.pkl", compress=3)

def now_brazil():
    # the training timestamps are Brazilian local time
    try:
        from zoneinfo import ZoneInfo
        return datetime.now(ZoneInfo("America/Sao_Paulo")).replace(tzinfo=None)
    except Exception:
        return datetime.now()

def predict_delivery(category, state, bundle, when=None):
    # category + delivery state -> estimated delivery date, counted from the click
    t = bundle["typical"]
    if category not in t["product"].index or state not in t["state"].index:
        raise ValueError("Unknown category or state")
    when = when or now_brazil()
    row = pd.DataFrame([make_row(t, category, state, when.month, when.weekday(), when.hour)])[feature_cols]
    days = max(float(bundle["model"].predict(row)[0]), 1.0)
    return {"days": round(days, 1), "date": (when + timedelta(days=round(days))).date(),
            "typical_error_days": round(bundle["typical_error_days"], 1)}

print("saved delivery_model.pkl")

saved delivery_model.pkl


In [9]:
bundle = joblib.load("delivery_model.pkl")                    # what the app does once at start-up
pd.DataFrame([{"category": c, "state": s, **predict_delivery(c, s, bundle, when=datetime(2026, 10, 16, 12, 0))}
              for c in ["health_beauty", "furniture_decor"] for s in ["SP", "BA", "AM"]])

,category,state,days,date,typical_error_days
0,health_beauty,SP,8.1,2026-10-24,5.3
1,health_beauty,BA,18.2,2026-11-03,5.3
2,health_beauty,AM,25.5,2026-11-10,5.3
3,furniture_decor,SP,12.3,2026-10-28,5.3
4,furniture_decor,BA,20.6,2026-11-06,5.3
5,furniture_decor,AM,25.9,2026-11-11,5.3
